# CCA2 – CIFAR-10 CNN in **TensorFlow / Keras**
**Before running:** menu *Runtime → Change runtime type → T4 GPU → Save*. Then *Runtime → Run all*.

At the end, a file `results_tensorflow.json` downloads automatically, send it back to Claude.
**Screenshot** the output of the *Training* cell (Figure 2 of the report).

In [ ]:
# Step 1: Environment info
import time, sys
t0 = time.time()
import tensorflow as tf
import keras
import_time = time.time() - t0
gpus = tf.config.list_physical_devices("GPU")
gpu_name = tf.config.experimental.get_device_details(gpus[0]).get("device_name", "GPU") if gpus else "None (CPU)"
print("Python     :", sys.version.split()[0])
print("TensorFlow :", tf.__version__, "| Keras", keras.__version__)
print("Import time: %.2f s" % import_time)
print("Device     :", gpu_name)
if gpu_name.startswith("None"): print("!!! NO GPU - go to Runtime > Change runtime type > T4 GPU, then Run all again !!!")

In [ ]:
# Step 2: Load CIFAR-10 and split 45,000 train / 5,000 val / 10,000 test
import numpy as np
tf.keras.utils.set_random_seed(42)

t0 = time.time()
(x_full, y_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_full = x_full.astype("float32") / 255.0      # NHWC, scaled 0-1
x_test = x_test.astype("float32") / 255.0
y_full, y_test = y_full.flatten(), y_test.flatten()
x_train, y_train = x_full[:45000], y_full[:45000]
x_val,   y_val   = x_full[45000:], y_full[45000:]
data_time = time.time() - t0

BATCH, EPOCHS, LR = 64, 10, 1e-3
print("Train/Val/Test:", len(x_train), len(x_val), len(x_test), "| data load %.1f s" % data_time)

In [ ]:
# Step 3: Model, loss, optimizer   (KEY CODE for the report)
from tensorflow.keras import layers
model = tf.keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),  layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),  layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10)])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
model.summary()
n_params = model.count_params()

In [ ]:
# Step 4: Training   (SCREENSHOT THIS CELL'S OUTPUT -> Figure 2)
class EpochTimer(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self.t = time.time()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.time() - self.t)

timer = EpochTimer()
start = time.time()
h = model.fit(x_train, y_train, validation_data=(x_val, y_val),
              batch_size=BATCH, epochs=EPOCHS, shuffle=True, callbacks=[timer], verbose=1)
train_time = time.time() - start
print("Total training time: %.1f s" % train_time)

In [ ]:
# Step 5: Test accuracy, inference speed, memory
test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
model.predict(x_test[:256], verbose=0)            # warm-up
t = time.time()
logits = model.predict(x_test, batch_size=256, verbose=0)
infer_time = time.time() - t
preds = logits.argmax(1)
try:
    peak_mem = tf.config.experimental.get_memory_info("GPU:0")["peak"] / 1e6
except Exception:
    peak_mem = None
print("Test loss %.4f | Test accuracy %.4f" % (test_loss, test_acc))
print("Inference: %.2f s for 10,000 images = %.4f ms/image" % (infer_time, infer_time / 10000 * 1000))
print("Peak GPU memory:", None if peak_mem is None else "%.0f MB" % peak_mem)

In [ ]:
# Step 5b: Fresh-install test for the 'Installation and setup' comparison (max 10 min, then skipped)
# Installs TensorFlow into a separate folder (does not touch the running environment)
import subprocess, shutil, os
install_time = install_size_mb = None
try:
    shutil.rmtree("/tmp/fresh_pkgs", ignore_errors=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                        "--target", "/tmp/fresh_pkgs", "tensorflow"], capture_output=True, text=True, timeout=600)
    install_time = time.time() - t0
    install_size_mb = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk("/tmp/fresh_pkgs") for f in fs) / 1e6
    print("Fresh install time: %.0f s | size on disk: %.0f MB" % (install_time, install_size_mb))
    if r.returncode != 0: print(r.stderr[-1500:])
    shutil.rmtree("/tmp/fresh_pkgs", ignore_errors=True)
except Exception as e:
    print("Install test skipped:", e)

In [ ]:
# Step 6: Plots + save results (send results_tensorflow.json to Claude)
import json, matplotlib.pyplot as plt
history = {k: [float(v) for v in vals] for k, vals in h.history.items()}
history["epoch_time"] = timer.times
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ep = range(1, EPOCHS + 1)
ax[0].plot(ep, history["accuracy"], label="train"); ax[0].plot(ep, history["val_accuracy"], label="val"); ax[0].set_title("TensorFlow accuracy")
ax[1].plot(ep, history["loss"], label="train");     ax[1].plot(ep, history["val_loss"], label="val");     ax[1].set_title("TensorFlow loss")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.show()

results = {"framework": "TensorFlow", "framework_version": tf.__version__, "keras_version": keras.__version__,
           "python": sys.version.split()[0], "device": gpu_name, "import_time_s": import_time, "install_time_s": install_time, "install_size_mb": install_size_mb,
           "data_load_s": data_time, "params": int(n_params), "batch": BATCH, "epochs": EPOCHS, "lr": LR,
           "history": history, "train_time_s": train_time, "test_loss": float(test_loss),
           "test_acc": float(test_acc), "inference_total_s": infer_time,
           "inference_ms_per_image": infer_time / 10000 * 1000, "peak_mem_mb": peak_mem,
           "preds": preds.tolist(), "labels": y_test.tolist()}
with open("results_tensorflow.json", "w") as f: json.dump(results, f)
try:
    from google.colab import files; files.download("results_tensorflow.json")
except Exception as e:
    print("Download manually from the Files panel:", e)